# CECS 385 - QAOA ETF Scaling Test

I am comparing simulated QAOA with exhaustive and greedy search as the number of ETFs increases from 6 to 10, 15 and 20. Each method selects 3 ETFs using the same January 2025 return estimates and risk formula.

I want to measure how often QAOA reaches the best score, how far away it is when it misses, and how long the simulation takes. Exhaustive search gives the exact score to compare against.

The same QAOA settings and ten random seeds are used for each ETF count. This is a simulated optimization test, not a comparison of actual monthly or yearly investment returns.

## 1. Install packages

In [ ]:
%pip -q install qiskit qiskit-aer scikit-learn pandas numpy matplotlib scipy

## 2. Load the same January 2025 market inputs as Notebook 3

In [ ]:
import time
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
url = "https://raw.githubusercontent.com/manisahni/marketdata/main/daily_adjclose.csv"
TICKERS = [
    "XLK", "XLV", "XLF", "XLE", "XLI",
    "XLP", "XLU", "XLB", "XLY", "SPY",
    "IWM", "EFA", "EEM", "TLT", "IEF",
    "GLD", "LQD", "VNQ", "TIP", "HYG"
]

CHOOSE = 3
LAMBDA = 1.0
QUICK_TEST = True
SHOTS = 128 if QUICK_TEST else 1024
MAX_ITER = 12 if QUICK_TEST else 75
DEPTH = 1 if QUICK_TEST else 2
SEED = 7
CIRCUIT_SCALE = 100.0  # magnifies small decimal return scores for quantum phase gates

print("ETFs:", len(TICKERS), " | portfolios:", len(list(itertools.combinations(TICKERS, CHOOSE))))

In [ ]:

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

url = "https://raw.githubusercontent.com/manisahni/marketdata/main/daily_adjclose.csv"

data = pd.read_csv(url, index_col="Date", parse_dates=True)
data = data.sort_index()

missing = [ticker for ticker in TICKERS if ticker not in data.columns]

if missing:
    raise ValueError(f"Missing ETFs in GitHub dataset: {missing}")

prices = data.loc["2017-01-01":"2025-01-31", TICKERS].copy()

# prices before an ETF existed
prices = prices.ffill()

monthly = prices.resample("ME").last()
monthly = monthly.loc["2019-01-31":"2025-01-31"]

if monthly.loc["2020-01-31":"2025-01-31"].isna().any().any():
    raise ValueError("Missing ETF prices in the analysis period")

if (monthly <= 0).any().any():
    raise ValueError("ETF prices must be positive")

monthly_returns = monthly.pct_change(fill_method=None)

features = {}

for ticker in TICKERS:
    r = monthly_returns[ticker]

    features[ticker] = pd.DataFrame({
        "last_month": r,
        "last_3_months": monthly[ticker].pct_change(3),
        "last_6_months": monthly[ticker].pct_change(6),
        "volatility_3m": r.rolling(3).std(),
        "next_month": r.shift(-1)
    }).dropna()

predictions = []

columns = [
    "last_month",
    "last_3_months",
    "last_6_months",
    "volatility_3m"
]

for ticker in TICKERS:
    df = features[ticker]

    # Train only on examples whose targets are known by Dec 2024
    train = df.loc[df.index <= "2024-11-30"]

    r = monthly_returns[ticker]

    forecast_row = pd.DataFrame({
        "last_month": r,
        "last_3_months": monthly[ticker].pct_change(3),
        "last_6_months": monthly[ticker].pct_change(6),
        "volatility_3m": r.rolling(3).std()
    }).loc[[pd.Timestamp("2024-12-31")]]

    if len(train) < 24 or forecast_row.isna().any().any():
        raise ValueError(f"Not enough training data for {ticker}")

    model = RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=3,
        random_state=SEED
    )

    model.fit(train[columns], train["next_month"])

    predictions.append(float(model.predict(forecast_row[columns])[0]))

mu = np.array(predictions)

cov = monthly_returns.loc[
    "2022-01-31":"2024-12-31", TICKERS
].cov().values

actual_jan = monthly_returns.loc[
    pd.Timestamp("2025-01-31"), TICKERS
].values

assert np.isfinite(mu).all()
assert np.isfinite(cov).all()
assert np.isfinite(actual_jan).all()

print("DATA SOURCE: GitHub - manisahni/marketdata")
print("ETFs:", len(TICKERS))
print("Monthly observations:", len(monthly))
print("Portfolios:", 1140)

print("\nJanuary 2025 ETF predictions:")

display(pd.DataFrame({
    "ETF": TICKERS,
    "Predicted Jan return": mu,
    "Actual Jan return": actual_jan
}).round(4))


## Settings

Each ETF count uses QAOA depth 2, 50 COBYLA iterations, 128 shots during optimization and 1,024 final shots. We run 10 trials for each of the four ETF counts.

For a short setup check, change `TRIAL_SEEDS` to `range(1)`. Change it back before collecting results.

In [ ]:
from pathlib import Path

ETF_COUNTS = (6, 10, 15, 20)
TRIAL_SEEDS = tuple(range(10))
QAOA_DEPTH = 2
QAOA_ITERATIONS = 50
OPTIMIZATION_SHOTS = 128
FINAL_SHOTS = 1024
MIXER_SCALE = CIRCUIT_SCALE
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

assert CHOOSE == 3 and len(TICKERS) == 20
assert ETF_COUNTS[-1] == len(TICKERS)
assert all(3 <= n <= len(TICKERS) for n in ETF_COUNTS)
assert len(set(TRIAL_SEEDS)) == len(TRIAL_SEEDS)
assert QAOA_DEPTH >= 1 and QAOA_ITERATIONS >= 1

print("ETF counts:", ETF_COUNTS)
print("QAOA depth:", QAOA_DEPTH, "| starts per size:", len(TRIAL_SEEDS))
print("Optimizer:", "COBYLA", "| max iterations:", QAOA_ITERATIONS)
print("Optimization shots:", OPTIMIZATION_SHOTS, "| final shots:", FINAL_SHOTS)

## 3. Define exact search, greedy search, and QAOA

In [ ]:
def make_problem(n):
    names = TICKERS[:n]
    returns = np.asarray(mu[:n], dtype=float)
    covariance = np.asarray(cov[:n, :n], dtype=float)
    assert len(names) == n
    assert np.isfinite(returns).all() and np.isfinite(covariance).all()

    def score(picks):
        w = np.zeros(n)
        w[list(picks)] = 1.0 / CHOOSE
        return float(returns @ w - LAMBDA * (w @ covariance @ w))

    exact_start = time.perf_counter()
    candidates = list(itertools.combinations(range(n), CHOOSE))
    values = np.array([score(picks) for picks in candidates])
    best_index = int(np.argmax(values))
    exact = candidates[best_index]
    exact_score = float(values[best_index])
    score_range = float(np.ptp(values))
    exact_seconds = time.perf_counter() - exact_start

    greedy_start = time.perf_counter()
    greedy = []
    for _ in range(CHOOSE):
        options = (i for i in range(n) if i not in greedy)
        greedy.append(max(options, key=lambda i: score(greedy + [i])))
    greedy_seconds = time.perf_counter() - greedy_start

    a = returns / CHOOSE - LAMBDA * np.diag(covariance) / (CHOOSE ** 2)
    b = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            b[i, j] = -2 * LAMBDA * covariance[i, j] / (CHOOSE ** 2)

    def encoded_score(picks):
        x = np.zeros(n)
        x[list(picks)] = 1.0
        return float(a @ x + sum(
            b[i, j] * x[i] * x[j]
            for i in range(n) for j in range(i + 1, n)
        ))

    max_error = max(abs(score(p) - encoded_score(p)) for p in candidates)
    assert max_error < 1e-10, "QAOA and classical scores do not match"
    assert exact_score + 1e-12 >= score(greedy)

    return {
        "n": n, "names": names, "score": score,
        "exact": exact, "exact_score": exact_score,
        "greedy": tuple(sorted(greedy)), "greedy_score": score(greedy),
        "score_range": score_range, "a": a, "b": b,
        "combinations": len(candidates), "encoding_error": max_error,
        "exact_seconds": exact_seconds, "greedy_seconds": greedy_seconds
    }


def qaoa_circuit(problem, angles):
    n = problem["n"]
    a = problem["a"]
    b = problem["b"]
    qc = QuantumCircuit(n)

    starting_picks = np.linspace(0, n - 1, CHOOSE, dtype=int)
    assert len(set(starting_picks)) == CHOOSE
    for i in starting_picks:
        qc.x(int(i))

    for gamma, beta in zip(angles[:QAOA_DEPTH], angles[QAOA_DEPTH:]):
        for i in range(n):
            adjustment = a[i] + 0.5 * sum(
                b[min(i, j), max(i, j)]
                for j in range(n) if j != i
            )
            qc.rz(MIXER_SCALE * gamma * adjustment, i)

        for i in range(n):
            for j in range(i + 1, n):
                if abs(b[i, j]) > 1e-12:
                    qc.rzz(-MIXER_SCALE * gamma * b[i, j] / 2, i, j)

        for offset in (0, 1):
            for i in range(offset, n - 1, 2):
                qc.rxx(beta, i, i + 1)
                qc.ryy(beta, i, i + 1)
        qc.rxx(beta, n - 1, 0)
        qc.ryy(beta, n - 1, 0)

    qc.measure_all()
    return qc


def run_qaoa(problem, seed):
    n = problem["n"]
    rng = np.random.default_rng(seed)
    sample_rng = np.random.default_rng(seed + 10_000)
    simulator = AerSimulator(method="matrix_product_state")
    initial = rng.uniform(0, 2 * np.pi, 2 * QAOA_DEPTH)
    calls = [0]

    def get_samples(angles, shots):
        circuit = qaoa_circuit(problem, angles)
        compiled = transpile(circuit, simulator, optimization_level=1)
        result = simulator.run(
            compiled, shots=shots,
            seed_simulator=int(sample_rng.integers(0, 2**31 - 1))
        ).result()
        counts = result.get_counts()
        valid = []
        for bitstring, count in counts.items():
            picks = tuple(i for i, bit in enumerate(bitstring[::-1]) if bit == "1")
            if len(picks) == CHOOSE:
                valid.append((picks, count, problem["score"](picks)))
        return valid

    def loss(angles):
        samples = get_samples(angles, OPTIMIZATION_SHOTS)
        calls[0] += 1
        count = sum(item[1] for item in samples)
        if count == 0:
            return 1e4
        return -sum(item[1] * item[2] for item in samples) / count

    started = time.perf_counter()
    fitted = minimize(
        loss, initial, method="COBYLA",
        options={"maxiter": QAOA_ITERATIONS}
    )
    final = get_samples(fitted.x, FINAL_SHOTS)
    seconds = time.perf_counter() - started

    valid_shots = sum(item[1] for item in final)
    if not final:
        return {
            "etfs": n, "seed": seed, "depth": QAOA_DEPTH,
            "portfolio": "", "score": np.nan, "gap": np.nan,
            "normalized_gap": np.nan, "exact_hit": False,
            "valid_shots": 0, "runtime_seconds": seconds,
            "optimizer_calls": calls[0], "success": bool(fitted.success)
        }

    best_picks, _, best_score = max(final, key=lambda item: item[2])
    gap = problem["exact_score"] - best_score
    assert gap >= -1e-10, "QAOA exceeded the exhaustive optimum"
    assert len(best_picks) == CHOOSE
    assert 0 < valid_shots <= FINAL_SHOTS

    return {
        "etfs": n, "seed": seed, "depth": QAOA_DEPTH,
        "portfolio": ", ".join(problem["names"][i] for i in best_picks),
        "score": float(best_score),
        "gap": max(0.0, float(gap)),
        "normalized_gap": (
            max(0.0, float(gap)) / problem["score_range"]
            if problem["score_range"] > 1e-12 else 0.0
        ),
        "exact_hit": bool(np.isclose(best_score, problem["exact_score"], atol=1e-8, rtol=0)),
        "valid_shots": valid_shots, "runtime_seconds": seconds,
        "optimizer_calls": calls[0], "success": bool(fitted.success)
    }

## Check the setup

Check the portfolio counts and make sure the quantum and classical score calculations agree before running QAOA.

In [ ]:
expected_counts = {6: 20, 10: 120, 15: 455, 20: 1140}
problems = {n: make_problem(n) for n in ETF_COUNTS}

for n, p in problems.items():
    assert p["combinations"] == expected_counts[n]
    assert p["encoding_error"] < 1e-10
    assert p["exact_seconds"] >= 0 and p["greedy_seconds"] >= 0
    test_angles = np.zeros(2 * QAOA_DEPTH)
    circuit = qaoa_circuit(p, test_angles)
    assert circuit.num_qubits == n
    print(n, "ETFs:", p["combinations"], "portfolios; encoding checked")

print("Setup checks passed. No QAOA trials have been run yet.")

## Run QAOA

Run each ETF count using the same settings. The code saves the results after each trial. A larger ETF count may take longer to simulate.

In [ ]:
trial_rows = []
benchmark_rows = []

for n in ETF_COUNTS:
    p = problems[n]

    benchmark_rows.append({
        "etfs": n,
        "combinations": p["combinations"],
        "exact_portfolio": ", ".join(p["names"][i] for i in p["exact"]),
        "exact_score": p["exact_score"],
        "exact_seconds": p["exact_seconds"],
        "greedy_portfolio": ", ".join(p["names"][i] for i in p["greedy"]),
        "greedy_score": p["greedy_score"],
        "greedy_seconds": p["greedy_seconds"],
        "greedy_gap": p["exact_score"] - p["greedy_score"],
        "score_range": p["score_range"]
    })

    for seed in TRIAL_SEEDS:
        trial = run_qaoa(p, seed)
        trial_rows.append(trial)
        pd.DataFrame(trial_rows).to_csv(OUTPUT_DIR / "qaoa_size_trials.csv", index=False)
        print(
            f"{n} ETFs | seed {seed} | score {trial['score']:.6f} | "
            f"gap {trial['gap']:.6f} | runtime {trial['runtime_seconds']:.2f}s"
        )

benchmark_df = pd.DataFrame(benchmark_rows)
trials_df = pd.DataFrame(trial_rows)
benchmark_df.to_csv(OUTPUT_DIR / "qaoa_size_benchmarks.csv", index=False)

assert len(trials_df) == len(ETF_COUNTS) * len(TRIAL_SEEDS)
assert set(trials_df["etfs"]) == set(ETF_COUNTS)
assert (trials_df["valid_shots"] == FINAL_SHOTS).all(), "Some shots were invalid"

display(benchmark_df.round(6))
display(trials_df.round(6))

## Compare the results

- **Exact hit rate:** percentage of QAOA runs that found the best score.
- **Normalized gap:** distance from the best score relative to that ETF group's score range. Lower is better.
- **Runtime:** measured time in Python and the simulator, not quantum hardware cost.

In [ ]:
summary_df = trials_df.groupby("etfs", as_index=False).agg(
    runs=("seed", "count"),
    exact_hit_rate=("exact_hit", "mean"),
    average_normalized_gap=("normalized_gap", "mean"),
    average_gap=("gap", "mean"),
    average_runtime_seconds=("runtime_seconds", "mean"),
    avg_valid_shots=("valid_shots", "mean")
)
summary_df["exact_hit_rate"] *= 100
summary_df.to_csv(OUTPUT_DIR / "qaoa_size_summary.csv", index=False)
display(summary_df.round(5))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(summary_df["etfs"], summary_df["exact_hit_rate"], marker="o")
axes[0].set_ylabel("Exact optimal portfolios found (%)")
axes[0].set_title("Exact hit rate")
axes[1].plot(summary_df["etfs"], summary_df["average_normalized_gap"], marker="o")
axes[1].set_ylabel("Mean normalized gap (lower is better)")
axes[1].set_title("Solution quality")
axes[2].plot(summary_df["etfs"], summary_df["average_runtime_seconds"], marker="o")
axes[2].set_ylabel("Average simulated runtime (seconds)")
axes[2].set_title("QAOA runtime")

for ax in axes:
    ax.set_xticks(ETF_COUNTS)
    ax.set_xlabel("Number of ETFs")
    ax.grid(alpha=0.3)

fig.suptitle("QAOA with 3 ETFs selected; depth 2; same settings for each size")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "qaoa_size_scaling.png", dpi=170, bbox_inches="tight")
plt.show()

## Limitations

The ETF groups are nested, so increasing the group also changes which ETFs are included. I am using one January 2025 decision rather than several market periods.

The results can show how this simulated QAOA setup performs across ETF counts, but they cannot establish a real quantum speed or investment-return advantage. Those comparisons would need more dates, additional ETF groups and matched costs.